In [41]:
%pip install natsort
import pandas as pd
from pathlib import Path
from scipy.optimize import least_squares
from IPython.display import display
import matplotlib.pyplot as plt
import plotly.express as px
import numpy as np
import re
from natsort import natsorted

# =============================================================================
# 1. 参数区
# =============================================================================
folder_path = Path(r"D:\毕设数据\20_export_pulse\20_export_pulse\METABatt_Sony_Murata_18650VTC6_007")
SOH_FILENAME_PATTERN = re.compile(r"BM\d+_(\d+(?:\.\d+)?)SOH\.parquet$", flags=re.IGNORECASE)


def extract_soh_from_filename(filename):
    match = SOH_FILENAME_PATTERN.search(filename.strip())
    if match is None:
        raise ValueError(f"无法从文件名解析 SOH: {filename}")
    return float(match.group(1))


SOC_ORDER = ["90%", "50%", "10%"]

CYCLE_ACTIVE_HOUR = 3.0
PAUSE_AFTER_CYCLE_HOUR = 4.5
REMOVE_PULSE_BEFORE_MIN = 60

# 相邻 cycle 起点之间的总间隔
CYCLE_PERIOD_HOUR = CYCLE_ACTIVE_HOUR + PAUSE_AFTER_CYCLE_HOUR
CYCLE_PERIOD_MIN = CYCLE_PERIOD_HOUR * 60

# 设置电流标准差
STD_LIMIT_1P5A = 0.1
STD_LIMIT_3A = 0.1

OUTPUT_COLUMNS = [
    "SOH",
    "SOC",
    "File",
    "Time",
    "Current_level",
    "Zustand",
    "ID"
]

Note: you may need to restart the kernel to use updated packages.


In [42]:
# =============================================================================
# 2. 读取 parquet
# =============================================================================

parquet_files = natsorted(list(folder_path.rglob("*.parquet")))

if len(parquet_files) == 0:
    raise FileNotFoundError(f"没有在文件夹中找到 parquet 文件: {folder_path}")

df_list = []

for file in parquet_files:
    temp = pd.read_parquet(file)
    temp["File"] = file.name
    temp["SOH"] = extract_soh_from_filename(file.name)

    df_list.append(temp)

df = pd.concat(df_list, ignore_index=True)

In [43]:
# =============================================================================
# 3. 按时间划分 SOC
# =============================================================================

df["Time"] = pd.to_datetime(df["Time"], utc=True, errors="coerce")
df["Current"] = pd.to_numeric(df["Current"], errors="coerce")

df = df.dropna(subset=["Time", "Current"]).copy()
df = df.sort_values("Time").reset_index(drop=True)

# 第一行就是 cycle 1 起点，SOC = 90%
df = df.sort_values(["File", "Time"]).reset_index(drop=True)

file_start_time = df.groupby("File")["Time"].transform("min")

df["elapsed_min"] = (
    df["Time"] - file_start_time
).dt.total_seconds() / 60

df["cycle_id"] = (df["elapsed_min"] // CYCLE_PERIOD_MIN).astype(int) + 1

df["SOC"] = df["cycle_id"].apply(
    lambda x: SOC_ORDER[(x - 1) % len(SOC_ORDER)]
)

df["Current_level"] = df["Current"]

In [44]:
# =============================================================================
# 4. 只保留读取期间的脉冲
# =============================================================================

df["Zustand"] = df["Zustand"].astype(str)
df["Zustand_raw"] = df["Zustand"].astype(str)
df["Zustand"] = df["Zustand_raw"]
df["Zustand_upper"] = df["Zustand_raw"].str.upper()
df.loc[df["Zustand_upper"].str.startswith("DCH", na=False), "Zustand"] = "DCH"
df.loc[df["Zustand_upper"].str.startswith("CHA", na=False), "Zustand"] = "CHA"
df["Zustand_upper"] = df["Zustand"].str.upper()

df["pulse_segment_id"] = (
    df["File"].ne(df["File"].shift())
    | df["Zustand_upper"].ne(df["Zustand_upper"].shift())
).cumsum()

pulse_mask = df["Zustand_upper"].str.startswith(("CHA", "DCH"), na=False)

pulse_sequence = df[pulse_mask].copy()
pulse_sequence = pulse_sequence.sort_values(["File", "pulse_segment_id", "Time"])

# 先剔除电流不恒定的脉冲段
bad_segments = (
    pulse_sequence
    .groupby(["File", "pulse_segment_id"])["Current_level"]
    .nunique()
)

# 先剔除电流不稳定的脉冲段
# 如果首个测量点 Current_level == 0，则忽略首点，
# 从第二个测量点开始计算标准差
def is_bad_current_segment(group):
    group = group.sort_values("Time")

    current_values = group["Current"]

    if len(current_values) >= 2 and group["Current_level"].iloc[0] == 0:
        current_values = current_values.iloc[1:]

    current_abs_mean = current_values.abs().mean()
    current_std = current_values.std()

    # 判断是 1.5A 还是 3A pulse
    if abs(current_abs_mean - 1.5) < abs(current_abs_mean - 3.0):
        return current_std > STD_LIMIT_1P5A
    else:
        return current_std > STD_LIMIT_3A


bad_segments = (
    pulse_sequence
    .groupby(["File", "pulse_segment_id"])
    .filter(is_bad_current_segment)
    [["File", "pulse_segment_id"]]
    .drop_duplicates()
)

pulse_sequence = pulse_sequence.merge(
    bad_segments.assign(to_drop=True),
    on=["File", "pulse_segment_id"],
    how="left"
)

pulse_sequence = pulse_sequence[
    pulse_sequence["to_drop"].isna()
].drop(columns="to_drop")

def pick_pulse_row(group):
    group = group.sort_values("Time")

    # 如果该脉冲段第一个测量点 Current_level 为 0，
    # 就改用第二个测量点记录
    if group["Current_level"].iloc[0] == 0 and len(group) >= 2:
        return group.iloc[1]

    return group.iloc[0]

pulse_sequence = (
    pulse_sequence
    .groupby(["File", "pulse_segment_id"], group_keys=False)
    .apply(pick_pulse_row)
    .reset_index(drop=True)
)

# 最后再剔除 DCH 且 abs(Current_level) == 1.5
pulse_sequence = pulse_sequence[
    ~(
        pulse_sequence["Zustand_upper"].str.startswith("DCH", na=False)
        & pulse_sequence["Current_level"].abs().eq(1.5)
    )
].copy()

# 剔除每个 cycle 前 1h 内 pulse
pulse_sequence["time_in_cycle_min"] = (
    pulse_sequence["elapsed_min"] % CYCLE_PERIOD_MIN
)

pulse_sequence = pulse_sequence[
    pulse_sequence["time_in_cycle_min"] > REMOVE_PULSE_BEFORE_MIN
].copy()

pulse_sequence = pulse_sequence[OUTPUT_COLUMNS].copy()

display(pulse_sequence)

C:\Users\12639\AppData\Local\Temp\ipykernel_48372\3807771063.py:82: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(pick_pulse_row)


,SOH,SOC,File,Time,Current_level,Zustand,ID
1,90.3,90%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 13:04:35.510000+00:00,1.498165,CHA,12_21
2,90.3,90%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 14:05:18.700000+00:00,-2.995366,DCH,12_25
4,90.3,50%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 20:56:39.720000+00:00,1.499155,CHA,12_39
5,90.3,50%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 21:57:23.020000+00:00,-2.997524,DCH,12_43
6,90.3,50%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 22:58:26.330000+00:00,2.992477,CHA,12_47
...,...,...,...,...,...,...,...
256,92.2,50%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM9_9...,2024-10-23 22:29:49.320000+00:00,-2.998784,DCH,9_44
257,92.2,50%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM9_9...,2024-10-23 23:30:52.610000+00:00,2.999132,CHA,9_48
259,92.2,10%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM9_9...,2024-10-24 05:23:31.800000+00:00,1.496366,CHA,9_58
260,92.2,10%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM9_9...,2024-10-24 06:24:14.960000+00:00,-2.988168,DCH,9_62


In [45]:
# =============================================================================
# 5. 保存结果
# =============================================================================

output_folder = folder_path / "cycle_time_output"
output_folder.mkdir(exist_ok=True)

output_file = output_folder / "recognition_sequence_clean.csv"

pulse_sequence.to_csv(output_file, index=False, encoding="utf-8-sig")

print("保存完成:", output_file)

保存完成: D:\毕设数据\20_export_pulse\20_export_pulse\METABatt_Sony_Murata_18650VTC6_007\cycle_time_output\recognition_sequence_clean.csv
